# A Comparative and Explainable Machine Learning Study for Phishing Website Detection

This notebook contains the full experimental pipeline for the research project.

## Research Questions
- **RQ1:** Which machine learning model performs best for phishing website detection?
- **RQ2:** Which evaluation metrics are most informative for phishing detection?
- **RQ3:** Which website features contribute most to phishing classification?

## 1. Setup and Imports

In [ ]:
# Install dependencies (uncomment if running on Google Colab)
# !pip install ucimlrepo pandas numpy scikit-learn matplotlib seaborn

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)
from sklearn.inspection import permutation_importance
from sklearn.base import clone

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

print("All imports successful!")

## 2. Load Dataset

In [ ]:
# Fetch UCI Phishing Websites Dataset
phishing = fetch_ucirepo(id=327)

X = phishing.data.features.copy()
y_raw = phishing.data.targets.iloc[:, 0].copy()

print(f"Dataset shape: {X.shape}")
print(f"\nOriginal target distribution:")
print(y_raw.value_counts())

In [ ]:
# Convert target: Phishing = 1, Legitimate = 0
# Original: -1 = phishing, 1 = legitimate
y = (y_raw == -1).astype(int)

print("Binary target distribution:")
print(y.value_counts())
print(f"\nPhishing ratio: {y.mean()*100:.1f}%")

In [ ]:
# Explore features
print(f"Features ({X.shape[1]}):")
print(list(X.columns))
print(f"\nFirst 5 rows:")
X.head()

## 3. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTrain phishing ratio: {y_train.mean()*100:.1f}%")
print(f"Test phishing ratio: {y_test.mean()*100:.1f}%")

## 4. Define Models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=42
        ))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),

    "SVM-RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf", probability=True,
            class_weight="balanced", random_state=42
        ))
    ]),

    "Decision Tree": Pipeline([
        ("model", DecisionTreeClassifier(
            class_weight="balanced", random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("model", GradientBoostingClassifier(random_state=42))
    ])
}

print(f"Models defined: {list(models.keys())}")

## 5. Train and Evaluate on Test Set

In [ ]:
results = {}
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_score)

    results[name] = {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-score": f1,
        "ROC-AUC": auc
    }

    print(f"  -> Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

results_df = pd.DataFrame(results).T.sort_values(by="F1-score", ascending=False)
print("\n" + "="*60)
print("TEST SET RESULTS (sorted by F1-score):")
print("="*60)
results_df

In [ ]:
# Save test results
os.makedirs("../results", exist_ok=True)
results_df.to_csv("../results/test_results.csv")
print("Test results saved to results/test_results.csv")

## 6. Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = {}

for name, model in models.items():
    print(f"Cross-validating {name}...")

    scores = cross_validate(
        model, X_train, y_train,
        cv=cv, scoring=scoring, n_jobs=-1
    )

    cv_results[name] = {
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1-score": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    }

cv_results_df = pd.DataFrame(cv_results).T.sort_values(by="CV F1-score", ascending=False)
cv_results_df.to_csv("../results/cv_results.csv")

print("\nCROSS-VALIDATION RESULTS:")
cv_results_df

## 7. Best Model Selection

In [ ]:
best_model_name = results_df.index[0]
best_model = trained_models[best_model_name]

print(f"Best model based on F1-score: {best_model_name}")
print(f"\nPerformance:")
print(results_df.loc[best_model_name])

## 8. Confusion Matrix

In [ ]:
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Legitimate", "Phishing"],
    yticklabels=["Legitimate", "Phishing"]
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.tight_layout()
plt.savefig("../results/confusion_matrix.png", dpi=300)
plt.show()

print(f"\nClassification Report ({best_model_name}):")
print(classification_report(
    y_test, y_pred_best,
    target_names=["Legitimate", "Phishing"]
))

## 9. Permutation Feature Importance

In [ ]:
print(f"Computing permutation feature importance for {best_model_name}...")
print("(This may take a minute)")

perm = permutation_importance(
    best_model, X_test, y_test,
    scoring="f1",
    n_repeats=20,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values(by="Importance", ascending=False)

importance_df.to_csv("../results/feature_importance.csv", index=False)

print("\nTop 10 Most Important Features:")
importance_df.head(10)

In [ ]:
# Plot feature importance
top_n = 10
top_features = importance_df.head(top_n)

plt.figure(figsize=(8, 5))
sns.barplot(
    data=top_features,
    x="Importance", y="Feature",
    palette="viridis"
)
plt.title(f"Top {top_n} Most Important Features - {best_model_name}")
plt.xlabel("Permutation Importance (F1-score)")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("../results/feature_importance.png", dpi=300)
plt.show()

## 10. Feature Subset Analysis (Ablation Study)

In [ ]:
top_k_results = {}
total_features = X.shape[1]

for k in [5, 10, 15, total_features]:
    selected_features = importance_df.head(k)["Feature"].tolist()

    model_k = clone(best_model)
    model_k.fit(X_train[selected_features], y_train)

    y_pred_k = model_k.predict(X_test[selected_features])

    if hasattr(model_k, "predict_proba"):
        y_score_k = model_k.predict_proba(X_test[selected_features])[:, 1]
    else:
        y_score_k = model_k.decision_function(X_test[selected_features])

    label = f"Top {k}" if k < total_features else f"All {k}"

    top_k_results[label] = {
        "Accuracy": accuracy_score(y_test, y_pred_k),
        "Precision": precision_score(y_test, y_pred_k),
        "Recall": recall_score(y_test, y_pred_k),
        "F1-score": f1_score(y_test, y_pred_k),
        "ROC-AUC": roc_auc_score(y_test, y_score_k)
    }

top_k_df = pd.DataFrame(top_k_results).T
top_k_df.to_csv("../results/top_k_feature_results.csv")

print(f"Feature Subset Analysis ({best_model_name}):")
top_k_df

## 11. Summary

### Key Findings
- Fill in after running the experiments
- Best model: [BEST MODEL]
- Most important features: [TOP FEATURES]
- Feature subset performance: [OBSERVATIONS]